In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.weightstats import ttest_ind
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.api as sms
import statsmodels.api as sm
import os

In [11]:
#exclude collocated snesors that were already calibrated
exclude_sensors = {1, 9, 13, 25}
sensor_dfs = {}

for sensor_num in range(1, 26):
    if sensor_num in exclude_sensors:
        continue

    sensor_name = f"Sensor{sensor_num:02d}"
    file_path = f"NC raw beacon data/{sensor_name}.csv"

    df = pd.read_csv(file_path)

    df = (
        df[
            [
                "datetime",
                "co_wrk_aux",
                "no2_wrk_aux",
                "no_wrk_aux",
                "o3_wrk_aux",
                "rh",
                "temp",
            ]
        ]
        .rename(columns={"datetime": "datetime_utc"})
    )

    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"])

    sensor_dfs[sensor_name] = df

print(f"Loaded and processed {len(sensor_dfs)} sensors.")

Loaded and processed 21 sensors.


In [12]:
no2 = pd.read_csv("low RSD ready data/no2_bysite.csv")
no2["datetime_utc"] = pd.to_datetime(no2["datetime_utc"])

sensor_to_site = {
    "Sensor03": "dpw",
    "Sensor08": "dpw",
    "Sensor14": "dpw",
    "Sensor16": "dpw",
    "Sensor18": "dpw",
    "Sensor21": "dpw",

    "Sensor17": "mjf",

    "Sensor05": "pema",
    "Sensor06": "pema",
    "Sensor07": "pema",
    "Sensor15": "pema",
    "Sensor20": "pema",
    "Sensor24": "pema",

    "Sensor02": "pha",
    "Sensor04": "pha",
    "Sensor10": "pha",
    "Sensor11": "pha",
    "Sensor12": "pha",
    "Sensor19": "pha",
    "Sensor22": "pha",
    "Sensor23": "pha",
}

output_dir = "ML ready NC dpp no2 datasets"

sensor_train_dfs = {}


for sensor_name, site in sensor_to_site.items():

    ref_col_name = f"{site}_quant_no2"

    reference_df = (
        no2[["datetime_utc", site]]
        .rename(columns={site: ref_col_name})
    )

    beacon_df = sensor_dfs[sensor_name].copy()

    combined = pd.merge(beacon_df, reference_df, on="datetime_utc", how="inner")

    combined = combined.dropna(axis=0)
    combined = combined.sort_values("datetime_utc").reset_index(drop=True)

    sensor_train_dfs[sensor_name] = combined

    start_date = combined["datetime_utc"].min()
    end_date = combined["datetime_utc"].max()
    n_obs = len(combined)

    out_path = f"{output_dir}/{sensor_name}test.csv"
    combined.to_csv(out_path, index=False)

    print(f"{sensor_name}: {n_obs} rows | {start_date} → {end_date}")

Sensor03: 5607 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor08: 5608 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor14: 5607 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor16: 5610 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor18: 5606 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor21: 4082 rows | 2025-10-09 22:00:00 → 2026-03-29 06:00:00
Sensor17: 5606 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor05: 5192 rows | 2025-10-09 23:00:00 → 2026-05-31 23:00:00
Sensor06: 3986 rows | 2025-10-09 23:00:00 → 2026-05-31 23:00:00
Sensor07: 5197 rows | 2025-10-09 23:00:00 → 2026-05-31 23:00:00
Sensor15: 4754 rows | 2025-10-09 23:00:00 → 2026-05-28 15:00:00
Sensor20: 5196 rows | 2025-10-09 23:00:00 → 2026-05-31 23:00:00
Sensor24: 5150 rows | 2025-10-09 23:00:00 → 2026-05-31 23:00:00
Sensor02: 5613 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor04: 5587 rows | 2025-10-09 22:00:00 → 2026-05-31 23:00:00
Sensor10: 4763 rows | 2025-10-09 22:00:0

In [13]:
low_rsd = pd.read_csv("low RSD data/no2_low_rsd_dpp.csv")
low_rsd["datetime_utc"] = pd.to_datetime(low_rsd["datetime_utc"])

low_rsd_times = low_rsd["datetime_utc"]

train_dir = "ML ready NC dpp no2 datasets"

sensor_low_rsd_dfs = {}

for sensor_name, site in sensor_to_site.items():
    combined = sensor_train_dfs[sensor_name].copy()

    low_rsd_df = combined[
        combined["datetime_utc"].isin(low_rsd_times)
    ].copy()

    low_rsd_df = low_rsd_df.sort_values("datetime_utc").reset_index(drop=True)

    sensor_low_rsd_dfs[sensor_name] = low_rsd_df

    out_path = f"{train_dir}/{sensor_name}train.csv"
    low_rsd_df.to_csv(out_path, index=False)

    print(f"{sensor_name}: {len(low_rsd_df)} rows retained")

Sensor03: 521 rows retained
Sensor08: 521 rows retained
Sensor14: 520 rows retained
Sensor16: 521 rows retained
Sensor18: 521 rows retained
Sensor21: 395 rows retained
Sensor17: 520 rows retained
Sensor05: 521 rows retained
Sensor06: 414 rows retained
Sensor07: 521 rows retained
Sensor15: 464 rows retained
Sensor20: 521 rows retained
Sensor24: 507 rows retained
Sensor02: 521 rows retained
Sensor04: 521 rows retained
Sensor10: 460 rows retained
Sensor11: 0 rows retained
Sensor12: 521 rows retained
Sensor19: 459 rows retained
Sensor22: 464 rows retained
Sensor23: 406 rows retained
